In [22]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

Indexing : ( Document Ingestion )

In [23]:
# PDF path
pdf_path = r"C:\Users\a\Desktop\Output_Langchain\XYZ_Company_Policy_Demo.pdf"

# Load PDF
loader = PyPDFLoader(pdf_path)

documents = loader.load()

print(f"Number of pages: {len(documents)}")

Number of pages: 3


In [27]:
print(documents[0].page_content)

XYZ Company
 Employee Policies & Workplace Guidelines
Document Type: Internal Company Policy
Version: 2.0
Effective Date: January 1, 2026
1. Purpose
This policy handbook defines the basic workplace rules and expectations for employees of XYZ
Company. The purpose is to create a professional, respectful, secure, and productive working
environment.
2. Working Hours and Attendance
The standard working schedule is Monday through Friday, from 9:30 AM to 6:30 PM. Employees are
expected to be available during core hours from 10:00 AM to 5:00 PM. Employees working remotely
must remain reachable through approved communication tools during working hours.
If an employee expects to be late or absent, they should inform their reporting manager as early as
possible. Repeated unexplained absences or attendance issues may result in corrective action.
3. Leave Policy
XYZ Company provides paid annual leave, sick leave, and approved public holidays. Employees should
submit planned leave requests through t

In [26]:
transcript

"XYZ Company\n Employee Policies & Workplace Guidelines\nDocument Type: Internal Company Policy\nVersion: 2.0\nEffective Date: January 1, 2026\n1. Purpose\nThis policy handbook defines the basic workplace rules and expectations for employees of XYZ\nCompany. The purpose is to create a professional, respectful, secure, and productive working\nenvironment.\n2. Working Hours and Attendance\nThe standard working schedule is Monday through Friday, from 9:30 AM to 6:30 PM. Employees are\nexpected to be available during core hours from 10:00 AM to 5:00 PM. Employees working remotely\nmust remain reachable through approved communication tools during working hours.\nIf an employee expects to be late or absent, they should inform their reporting manager as early as\npossible. Repeated unexplained absences or attendance issues may result in corrective action.\n3. Leave Policy\nXYZ Company provides paid annual leave, sick leave, and approved public holidays. Employees should\nsubmit planned leave 

In [28]:
for doc in documents:
    print(doc.page_content)
    print("-" * 80)

XYZ Company
 Employee Policies & Workplace Guidelines
Document Type: Internal Company Policy
Version: 2.0
Effective Date: January 1, 2026
1. Purpose
This policy handbook defines the basic workplace rules and expectations for employees of XYZ
Company. The purpose is to create a professional, respectful, secure, and productive working
environment.
2. Working Hours and Attendance
The standard working schedule is Monday through Friday, from 9:30 AM to 6:30 PM. Employees are
expected to be available during core hours from 10:00 AM to 5:00 PM. Employees working remotely
must remain reachable through approved communication tools during working hours.
If an employee expects to be late or absent, they should inform their reporting manager as early as
possible. Repeated unexplained absences or attendance issues may result in corrective action.
3. Leave Policy
XYZ Company provides paid annual leave, sick leave, and approved public holidays. Employees should
submit planned leave requests through t

Text Splitter

In [29]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

In [30]:
len(chunks)

7

In [32]:
chunks[5].page_content

'responsibilities, quality of work, collaboration, reliability, communication, and progress toward agreed\nobjectives.\n11. Learning and Development\nXYZ Company encourages employees to improve their professional skills. Depending on role and\nbusiness requirements, the company may provide access to training programs, workshops, technical\ncourses, conferences, or learning platforms.\n12. Disciplinary Process\nPolicy violations may result in corrective or disciplinary action depending on the nature and severity of\nthe issue. Possible actions include coaching, written warnings, suspension, or termination, subject to\napplicable law and company procedures.\n13. Frequently Asked Questions\nQuestion\nPolicy Answer\nWhat are the standard working hours?\nMonday to Friday, 9:30 AM to 6:30 PM, with core hours from 10:00 AM to 5:00 PM.\nHow early should planned leave be requested?\nLeave longer than three consecutive working days should normally be requested at least seven calendar days in adv

Embedding

In [38]:
embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2"
)

Vector Store

In [39]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("FAISS vector store created successfully!")

FAISS vector store created successfully!


In [40]:
vectorstore.index_to_docstore_id

{0: 'eb6ffa67-baaa-4c7b-bb96-4572b16f6dc6',
 1: '8c0bf594-6457-49e0-a68f-a44e8299cf5c',
 2: '95b7589f-4340-42e4-bf31-a7ba848a66bd',
 3: 'bdae1da2-31ad-4950-97c0-a2d32c2e7d30',
 4: 'a5a21408-f940-40c1-a646-6483eeaf2cbf',
 5: 'ec7a1dc5-8de3-4be0-9efe-596469439897',
 6: '3b764bcd-5e61-430e-9953-2300d92292a4'}

In [41]:
vectorstore.get_by_ids(['a5a21408-f940-40c1-a646-6483eeaf2cbf'])

[Document(id='a5a21408-f940-40c1-a646-6483eeaf2cbf', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-31T11:07:55+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-31T11:07:55+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'C:\\Users\\a\\Desktop\\Output_Langchain\\XYZ_Company_Policy_Demo.pdf', 'total_pages': 3, 'page': 2, 'page_label': '3'}, page_content='7. Code of Conduct\nAll employees are expected to communicate respectfully and behave professionally. XYZ Company\ndoes not tolerate harassment, discrimination, bullying, threats, or deliberate intimidation in the workplace\nor through company communication channels.\n8. Confidentiality\nEmployees may have access to confidential business information. Such information must not be\ndisclosed to unauthorized individuals or used for personal benefit. Confidentiality obligations may\ncontinue after an employee le

Retriver

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2},
)

In [47]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001C276EBD550>, search_kwargs={'k': 2})

In [48]:
query = "What are the standard working hours at XYZ Company?"

results = retriever.invoke(query)

results

[Document(id='eb6ffa67-baaa-4c7b-bb96-4572b16f6dc6', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-31T11:07:55+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-31T11:07:55+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'C:\\Users\\a\\Desktop\\Output_Langchain\\XYZ_Company_Policy_Demo.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='XYZ Company\n Employee Policies & Workplace Guidelines\nDocument Type: Internal Company Policy\nVersion: 2.0\nEffective Date: January 1, 2026\n1. Purpose\nThis policy handbook defines the basic workplace rules and expectations for employees of XYZ\nCompany. The purpose is to create a professional, respectful, secure, and productive working\nenvironment.\n2. Working Hours and Attendance\nThe standard working schedule is Monday through Friday, from 9:30 AM to 6:30 PM. Employees are\nexpected to be available du

LLM Model Define

In [69]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

Prompt

In [61]:
prompt = PromptTemplate(
    template="""
You are an AI assistant answering questions about XYZ Company policies.

Use ONLY the context provided below to answer the question.
If the answer is not present in the context, say:
"I don't know based on the provided document."

Context:
{context}

Question:
{question}

Answer:
""",
    input_variables=["context", "question"]
)

Genaration

In [62]:
query = "How many days in advance should planned leave be requested?"

docs = retriever.invoke(query)

In [63]:
docs

[Document(id='3b764bcd-5e61-430e-9953-2300d92292a4', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-31T11:07:55+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-31T11:07:55+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'C:\\Users\\a\\Desktop\\Output_Langchain\\XYZ_Company_Policy_Demo.pdf', 'total_pages': 3, 'page': 2, 'page_label': '3'}, page_content='How early should planned leave be requested?\nLeave longer than three consecutive working days should normally be requested at least seven calendar days in advance.\nCan employees work remotely?\nEligible employees may work remotely when approved by their team and role.\nWho should receive security incident reports?\nThe IT or security team should be notified promptly.\nHow quickly should expenses be submitted?\nExpense claims should normally be submitted within 30 days.\n End of XYZ Company Policy Document'

In [64]:
context = "\n\n".join(doc.page_content for doc in docs)

In [65]:
context

"How early should planned leave be requested?\nLeave longer than three consecutive working days should normally be requested at least seven calendar days in advance.\nCan employees work remotely?\nEligible employees may work remotely when approved by their team and role.\nWho should receive security incident reports?\nThe IT or security team should be notified promptly.\nHow quickly should expenses be submitted?\nExpense claims should normally be submitted within 30 days.\n End of XYZ Company Policy Document\n\n3. Leave Policy\nXYZ Company provides paid annual leave, sick leave, and approved public holidays. Employees should\nsubmit planned leave requests through the company's designated HR system whenever possible.\nFor planned leave of more than three consecutive working days, employees should normally submit the\nrequest at least seven calendar days in advance. Emergency or sick leave should be communicated to\nthe manager as soon as practical.\n4. Remote Work Policy\nEligible emplo

In [66]:
final_prompt = prompt.format(
    context=context,
    question=query
)

In [67]:
final_prompt

'\nYou are an AI assistant answering questions about XYZ Company policies.\n\nUse ONLY the context provided below to answer the question.\nIf the answer is not present in the context, say:\n"I don\'t know based on the provided document."\n\nContext:\nHow early should planned leave be requested?\nLeave longer than three consecutive working days should normally be requested at least seven calendar days in advance.\nCan employees work remotely?\nEligible employees may work remotely when approved by their team and role.\nWho should receive security incident reports?\nThe IT or security team should be notified promptly.\nHow quickly should expenses be submitted?\nExpense claims should normally be submitted within 30 days.\n End of XYZ Company Policy Document\n\n3. Leave Policy\nXYZ Company provides paid annual leave, sick leave, and approved public holidays. Employees should\nsubmit planned leave requests through the company\'s designated HR system whenever possible.\nFor planned leave of m

In [70]:
response = llm.invoke(final_prompt)

print(response.content)

Seven calendar days in advance.


Chain

In [71]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [72]:
def format_docs(docs):
    context = "\n\n".join(doc.page_content for doc in docs)
    return context

In [74]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [75]:
parser = StrOutputParser()

In [76]:
main_chain = parallel_chain | prompt | llm | parser

In [77]:
main_chain.invoke(query)

'Seven calendar days in advance.'

In [78]:
main_chain.get_graph().print_ascii()

           +---------------------------------+         
           | Parallel<context,question>Input |         
           +---------------------------------+         
                    **               ***               
                 ***                    **             
               **                         ***          
+----------------------+                     **        
| VectorStoreRetriever |                      *        
+----------------------+                      *        
            *                                 *        
            *                                 *        
            *                                 *        
    +-------------+                   +-------------+  
    | format_docs |                   | Passthrough |  
    +-------------+*                  +-------------+  
                    **               **                
                      ***         ***                  
                         **     **              